# D1.4 · Detection engineering *for* agents

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *Security of AI*

---

**Risk.** Scope drift, unusual tool sequencing, off-hours autonomous action.

**Control.** Detections whose subject is a non-human principal.

**This lab.** Five detections whose subject is a non-human principal.

| | |
|---|---|
| Open-source tooling | Falco, Sigma |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D1.4"))

Detection engineering *for* agents is the new work. The classic behavioural baselines invert: what is anomalous for a person is normal for a loop, and vice versa.

In [ ]:
from cybercommons import soc
import time

now = time.time()
HUMAN_BASELINES = {
 "logins from two countries in an hour": "incident for a person, routine for a service",
 "300 file reads per minute":            "incident for a person, idle for an agent",
 "activity at 03:00":                    "suspicious for a person, meaningless for an agent",
 "the same action 500 times":            "suspicious for a person, a stuck loop for an agent",
}
for k, v in HUMAN_BASELINES.items():
    print(f"  {k:40s} {v}")

print("\nAgent-appropriate signals instead:")
AGENT_SIGNALS = {
 "tool mix changed from baseline": "new capability or new prompt — re-test controls",
 "action rate dropped to zero":    "the loop is stuck or was killed",
 "a tool never seen before":       "manifest changed without review",
 "scope used exceeds scope needed": "over-granted identity",
}
for k, v in AGENT_SIGNALS.items():
    print(f"  {k:40s} {v}")

Now build one of them, because 'tool mix changed' is the highest-yield agent detection and almost nobody has it.

In [ ]:
base = soc.Baseline(tool_mix={"read_file": 0.80, "http_get": 0.15,
                                "write_file": 0.05}, actions_per_hour=400)
today = ([soc.Event(now, "patch-agent", "read_file")] * 40 +
         [soc.Event(now, "patch-agent", "http_get")] * 10 +
         [soc.Event(now, "patch-agent", "run_shell")] * 30)   # new tool
d = base.compare(today)
for k, v in d.items():
    print(f"{k:12s} {v}")

### Expect

The inversion table prints, and the drift comparison reports significant drift with `run_shell` listed as a new tool the baseline never contained.

### Your turn

Write the alert text for that drift detection. It has to tell the analyst what changed and what to do — 'anomaly detected' fails both tests.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D1.4.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*